# 7.1 Meta Learning

The Twitter dataset comprises approximately 1,500 instances of Twitter users, including their posted tweets, Twitter names, and demographic information such as age and gender. 
Given the diverse data types present in the dataset (textual, numerical, categorical), it offers potential for training multiple models on different data types.

With the diverse data at hand, we want to develop a **Meta classifier that predicts the gender of Twitter users based on their tweets and user names.** Therefore, we are going to train three sub-models.

Let's start by importing pickled data from 'data/twitterData.pkl' and showing its shape.

In [1]:
from pathlib import Path

candidate_paths = [
  Path("configs/visualisations.mplstyle"),
  Path("07_Solution_7/configs/visualisations.mplstyle"),
]

vis_path = next((path for path in candidate_paths if path.exists()), None)

if vis_path is None:
  raise FileNotFoundError("Could not find configs/visualisations.mplstyle. Start Jupyter in the repository root or 07_Solution_7.")

candidate_paths = [
  parent / "data" / "twitterData.pkl"
  for parent in (Path.cwd(), *Path.cwd().parents)
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
  raise FileNotFoundError("Could not find data/twitterData.pkl in the repository.")

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

plt.style.use(str(vis_path.resolve()))

data = pd.read_pickle(str(data_path.resolve()))

print(f"Data shape: {data.shape}")
data.head()

Data shape: (2916, 6)


,screen_name,name,tweets_concatenated,avatar_url,gender,age
0,DatZerooo,David,Warum riecht mein Bruder nach Pizza wenn er ei...,http://pbs.twimg.com/profile_images/7569661512...,M,16.0
0,reap705,Oliver Gast,[CSS] Ein Off-canvas-Menü mit Dropdown-Navigat...,http://pbs.twimg.com/profile_images/1366984169...,M,15.0
0,Dossary1995,Khalid Aldossary,,http://pbs.twimg.com/profile_images/4167523242...,M,37.0
0,eduUu06,eduUu,heut abend kogge und morgen endlich haare ab :...,http://pbs.twimg.com/profile_images/896480580/...,M,46.0
0,Narutofreak935,Avengar,@GrandlineTV gib nicht auf und mach dein Ding ...,http://pbs.twimg.com/profile_images/7317971734...,F,19.0


## Preprocessing

### **TASK: Data Exploration and Cleaning**

The Twitter dataset comprises approximately 1,500 instances of Twitter users, including their posted tweets, Twitter names, and demographic information such as age and gender. Before building classifiers, we need to explore and clean the data.

Steps:
1. Check if there are any users without tweets
2. Report how often this occurs
3. Remove users without tweets from the dataset

In [3]:
# --- YOUR CODE GOES HERE (END) ---
instances_empty_tweets = data[data["tweets_concatenated"] == ""]
ratio_empty = len(instances_empty_tweets) / len(data) 
data = data[data["tweets_concatenated"] != ""]
# ---- YOUR CODE GOES HERE (START) ---

print(f"Number of users without tweets: {len(instances_empty_tweets)}")
print(f"Ratio of users without tweets: {ratio_empty:.2%}")

Number of users without tweets: 430
Ratio of users without tweets: 14.75%


### **TASK: Target Attributes**

Analyze the distribution of the target variable (gender) in our dataset and preprocess it for classification.

Steps:
1. Create a binary gender class column (0 = F, 1 = M)
2. Calculate and report the class distribution

In [4]:
# (1.) Create binary gender class
# ---- YOUR CODE GOES HERE (START) ---
# binarize gender: 0 for female, 1 for male
data['gender_class'] = data['gender'].map({'F': 0, 'M': 1}).astype(int)
# --- YOUR CODE GOES HERE (END) ---

# (2.) Report distribution
# ---- YOUR CODE GOES HERE (START) ---
gender_counts = data['gender_class'].value_counts()
print(f"\nGender distribution:")
print(f"  Female (0): {gender_counts[0]} ({gender_counts[0]/len(data):.1%})")
print(f"  Male (1): {gender_counts[1]} ({gender_counts[1]/len(data):.1%})")
# --- YOUR CODE GOES HERE (END) ---


Gender distribution:
  Female (0): 1132 (45.5%)
  Male (1): 1354 (54.5%)


### **TASK: Data Splitting for Meta Learning**

Meta learning requires three subsets: a sub-training set, meta-training set, and a test set. The sub-training set trains base classifiers, the meta-training set trains the meta classifier, and the test set evaluates the final performance.

Steps:
1. Split data into a training set for the individual classifiers (60 %), training set for the meta model (16 %) and a test set (24 %)
2. Extract tweets and names for each split
3. Extract target variable (gender_class) for each split

In [5]:
# (1.) Create three-way split
# ---- YOUR CODE GOES HERE (START) ---
# split data into 60% train_sub and 40% rest
train_sub, temp_data = train_test_split(data, test_size=0.4, random_state=42)
# split rest into 60% train_meta and 40% test (i.e., 24% and 16% of original data)
train_meta, test = train_test_split(temp_data, test_size=0.6, random_state=42)
# --- YOUR CODE GOES HERE (END) ---

print(f"Split sizes:")
print(f"  train_sub: {len(train_sub)} ({len(train_sub)/len(data):.1%})")
print(f"  train_meta: {len(train_meta)} ({len(train_meta)/len(data):.1%})")
print(f"  test: {len(test)} ({len(test)/len(data):.1%})")

# (2.) Extract features for each split
# ---- YOUR CODE GOES HERE (START) ---
train_sub_tweets = train_sub["tweets_concatenated"]
train_meta_tweets = train_meta["tweets_concatenated"]
test_tweets = test["tweets_concatenated"]

train_sub_names = train_sub["name"]
train_meta_names = train_meta["name"]
test_names = test["name"]
# --- YOUR CODE GOES HERE (END) ---

# (3.) Extract target variable
# ---- YOUR CODE GOES HERE (START) ---
y_train_sub = train_sub["gender_class"]
y_train_meta = train_meta["gender_class"]
y_test = test["gender_class"]
# --- YOUR CODE GOES HERE (END) ---

Split sizes:
  train_sub: 1491 (60.0%)
  train_meta: 398 (16.0%)
  test: 597 (24.0%)


## Sub-Models

### **TASK: Tweet-Based Classifier (Bag of Words)**

Build a Naive Bayes classifier using the Bag of Words approach on tweets. This classifier will serve as one of our base models for the meta learner. Use sklearn's CountVectorizer [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) and MultinomialNB [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

Steps:
1. Create and fit CountVectorizer on training tweets
2. Transform all tweet sets
3. Train Naive Bayes classifier
4. Evaluate on test set
5. Generate predictions for meta learner

In [6]:
# (1.) Vectorize tweets using Bag of Words
# ---- YOUR CODE GOES HERE (START) ---
# initialize CountVectorizer for tweets to convert text to Bag of Words representation
countvectorizer_tweets = CountVectorizer()
# fit and transform training subset tweets, transform meta training and test tweets
X_train_sub_tweets = countvectorizer_tweets.fit_transform(train_sub_tweets)
X_train_meta_tweets = countvectorizer_tweets.transform(train_meta_tweets)
# transform test tweets
X_test_tweets = countvectorizer_tweets.transform(test_tweets)
# --- YOUR CODE GOES HERE (END) ---

print(f"Vocabulary size: {len(countvectorizer_tweets.get_feature_names_out())}")
print(f"Example features: {countvectorizer_tweets.get_feature_names_out()[100:110]}")

# View transformed data (first 5 rows, first 10 features)
tweet_features_df = pd.DataFrame(
    X_train_sub_tweets.todense()[:5, :10],
    columns=countvectorizer_tweets.get_feature_names_out()[:10]
)
print("\nExample of Bag of Words representation:")
print(tweet_features_df)

# (2.) Train Naive Bayes on tweets
# ---- YOUR CODE GOES HERE (START) ---
# initialize and train Multinomial Naive Bayes classifier on tweet features
bayes_tweets = MultinomialNB()
bayes_tweets.fit(X_train_sub_tweets, y_train_sub)
# --- YOUR CODE GOES HERE (END) ---

# (3.) Evaluate on test set
# ---- YOUR CODE GOES HERE (START) ---
# compute accuracy on test tweet features
tweet_score = bayes_tweets.score(X_test_tweets, y_test)
# --- YOUR CODE GOES HERE (END) ---

print(f"\nTweet-based Classifier Accuracy: {tweet_score:.2%}")

# (4.) Generate predictions for meta learner
# ---- YOUR CODE GOES HERE (START) ---
meta_input_tweets = pd.Series(bayes_tweets.predict(X_train_meta_tweets))
test_input_tweets = pd.Series(bayes_tweets.predict(X_test_tweets))
# --- YOUR CODE GOES HERE (END) ---

Vocabulary size: 212825
Example features: ['0190' '0195' '01emvifn7q' '01g9ux4311' '01gulx8z' '01kg' '01l7b02svl'
 '01m' '01m7hsw3vy' '01okrkmhsa']

Example of Bag of Words representation:
   00  000  0000  00002  0001  000de  000e  000er  000g  000ghmezhk
0   0    0     0      0     0      0     0      0     0           0
1   1    0     0      0     0      0     0      0     0           0
2   0    0     0      0     0      0     0      0     0           0
3   0    0     0      0     0      0     0      0     0           0
4   0    0     0      0     0      0     0      0     0           0

Tweet-based Classifier Accuracy: 74.20%


### **TASK: Name-Based Classifier (Bag of Words)**
Build another Naive Bayes classifier, this time using names as features for the Bag of Words approach.

Steps:
1. Vectorize names using CountVectorizer
2. Train Naive Bayes classifier on names
3. Evaluate performance
4. Generate predictions for meta learner

In [7]:
# (1.) Vectorize names
# ---- YOUR CODE GOES HERE (START) ---
# initialize CountVectorizer for names using character n-grams (1 to 3).
# n-grams are sequences of n characters that are found in the text and help capture patterns in names better than word-level features.
countvectorizer_names = CountVectorizer(analyzer='char', ngram_range=(1, 3))
# fit and transform training subset names
X_train_sub_names = countvectorizer_names.fit_transform(train_sub_names)
X_train_meta_names = countvectorizer_names.transform(train_meta_names)
# transform meta training and test names
X_test_names = countvectorizer_names.transform(test_names)
# --- YOUR CODE GOES HERE (END) ---

print(f"Name features size: {len(countvectorizer_names.get_feature_names_out())}")
print(f"Example character n-grams: {countvectorizer_names.get_feature_names_out()[100:110]}")

# (2.) Train Naive Bayes on names
# ---- YOUR CODE GOES HERE (START) ---
# initialize and train Multinomial Naive Bayes classifier on name features
bayes_names = MultinomialNB()
bayes_names.fit(X_train_sub_names, y_train_sub)
# --- YOUR CODE GOES HERE (END) ---

# (3.) Evaluate performance
# ---- YOUR CODE GOES HERE (START) ---
# compute accuracy on test name features
name_score_ml = bayes_names.score(X_test_names, y_test)
# --- YOUR CODE GOES HERE (END) ---

print(f"\nName-based Classifier (ML) Accuracy: {name_score_ml:.2%}")

# (4.) Generate predictions for meta learner
# ---- YOUR CODE GOES HERE (START) ---
meta_input_names_ml = pd.Series(bayes_names.predict(X_train_meta_names))
test_input_names_ml = pd.Series(bayes_names.predict(X_test_names))
# --- YOUR CODE GOES HERE (END) ---

Name features size: 6285
Example character n-grams: [' d' ' d:' ' da' ' de' ' di' ' do' ' dr' ' d⚽' ' e' ' e ']

Name-based Classifier (ML) Accuracy: 74.87%


### **TASK: Name-Based Classifier (Gender Guesser)**

An alternative approach for the name-based classifier is the gender-guesser package. This rule-based approach uses a pre-existing database of names and their associated genders.

Steps:
1. Import and test the gender detector
2. Apply detector to test names
3. Map outputs to our binary format (0/1)
4. Evaluate performance
5. Generate predictions for meta learner

In [ ]:
# %pip install gender-guesser

In [9]:
import gender_guesser.detector as gender

# (1.) Initialize and test detector
# ---- YOUR CODE GOES HERE (START) ---
detector = gender.Detector()

# Test examples
test_names_examples = ["Kathrin", "Catherine", "Sonja", "Xian", "Hans", "Leonardo"]
for name in test_names_examples:
    print(f"{name}: {detector.get_gender(name)}")
# --- YOUR CODE GOES HERE (END) ---

# (2.) Apply to test set
# ---- YOUR CODE GOES HERE (START) ---
y_test_gender_guesser = test_names.apply(detector.get_gender)
# --- YOUR CODE GOES HERE (END) ---

# (3.) Map to binary format
# ---- YOUR CODE GOES HERE (START) ---
# define mapping from gender-guesser categories to binary classes
gender_mapping = {
    'female': 0,
    'mostly_female': 0,
    'male': 1,
    'mostly_male': 1,
    'unknown': 2,  # Will be treated as wrong prediction
    'andy': 2      # Androgynous names
}
# map predictions to binary classes
y_test_gender_guesser = y_test_gender_guesser.map(gender_mapping).astype(int)
# --- YOUR CODE GOES HERE (END) ---

print(f"\nGender Guesser output distribution:")
print(y_test_gender_guesser.value_counts())

# (4.) Evaluate performance
# ---- YOUR CODE GOES HERE (START) ---
# compute accuracy on test name features
name_score_rule = accuracy_score(y_test, y_test_gender_guesser)
# --- YOUR CODE GOES HERE (END) ---

print(f"\nName-based Classifier (Gender Guesser) Accuracy: {name_score_rule:.2%}")

# (5.) Generate predictions for meta learner
# ---- YOUR CODE GOES HERE (START) ---
# generate meta input features using gender-guesser
meta_input_gender_guesser = train_meta_names.apply(detector.get_gender).replace(gender_mapping)
test_input_gender_guesser = test_names.apply(detector.get_gender).replace(gender_mapping)

# reset indices to match other inputs
meta_input_gender_guesser.index = meta_input_tweets.index
test_input_gender_guesser.index = test_input_tweets.index
# --- YOUR CODE GOES HERE (END) ---

# show some examples
examples_df = pd.DataFrame({
    'Name': test_names.values[:10],
    'Actual': y_test.values[:10],
    'Predicted': y_test_gender_guesser.values[:10]
})
print("\nExample predictions:")
print(examples_df)

Kathrin: female
Catherine: female
Sonja: female
Xian: andy
Hans: male
Leonardo: male

Gender Guesser output distribution:
name
2    523
0     43
1     31
Name: count, dtype: int64

Name-based Classifier (Gender Guesser) Accuracy: 10.55%

Example predictions:
                   Name  Actual  Predicted
0   A M E L I E   L N T       0          2
1            Klaus Pfau       1          2
2          Sandra.Lochi       0          2
3     Dr. Volker Ludwig       1          2
4                Kajita       0          2
5  Stefan Kaltenbrunner       1          2
6                  Lisa       0          0
7            Thore Koch       1          2
8         Nicolas Wöhrl       1          2
9              Borni_Ka       1          2


## Meta-Model

### **TASK: Meta Classifier**

Finally, we can combine all three sub-models using a Random Forest meta classifier. The meta classifier learns an optimal weighting of the predictions from each base model.

Steps:
1. Prepare meta training & test data
2. Train meta classifier
3. Evaluate on test set
4. Compare all approaches. How would you rate the success of the meta model?

In [10]:
# (1.) Prepare meta training & test data
# ---- YOUR CODE GOES HERE (START) ---
meta_data_train = pd.DataFrame({
    'tweet_based_prediction_bow': meta_input_tweets,
    'name_based_prediction_bow': meta_input_names_ml,
    'name_based_prediction_gg': meta_input_gender_guesser
})

meta_data_test = pd.DataFrame({
    'tweet_based_prediction_bow': test_input_tweets,
    'name_based_prediction_bow': test_input_names_ml,
    'name_based_prediction_gg': test_input_gender_guesser
})
# --- YOUR CODE GOES HERE (END) ---

# (2.) Train meta classifier
# ---- YOUR CODE GOES HERE (START) ---
meta_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
meta_classifier.fit(meta_data_train, y_train_meta)
# --- YOUR CODE GOES HERE (END) ---

# (3.) Evaluate on test set
# ---- YOUR CODE GOES HERE (START) ---
meta_score = meta_classifier.score(meta_data_test, y_test)
y_pred_meta = meta_classifier.predict(meta_data_test)
# --- YOUR CODE GOES HERE (END) ---

# (4.) Compare all approaches
# ---- YOUR CODE GOES HERE (START) ---
print("\n" + "=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"Tweet-based Classifier:            {tweet_score:.2%}")
print(f"Name-based Classifier (Bag of Words):        {name_score_ml:.2%}")
print(f"Name-based Classifier (Gender Guesser):      {name_score_rule:.2%}")
print(f"Meta Classifier (Combined):        {meta_score:.2%}")
# --- YOUR CODE GOES HERE (END) ---
print("=" * 60)


PERFORMANCE COMPARISON
Tweet-based Classifier:            74.20%
Name-based Classifier (Bag of Words):        74.87%
Name-based Classifier (Gender Guesser):      10.55%
Meta Classifier (Combined):        75.71%


The two bag-of-words approaches already set a strong baseline. However, by combining the three models, we can squeeze out another percentage point. So the models seem to complement each other, at least to some extent.